# Runtime

## Agent State
* Every agent manages its execution context through AgentState, a typed dictionary that holds the current conversation history and any custom fields your tools and middleware need.


    <img src="../../assets/agent_state.png" width="400" height="300">

## Runtime
* LangChain’s `create_agent` runs on LangGraph’s runtime under the hood.
* LangGraph exposes a **Runtime** object with the following information:
    1. **Context**: Static, per run information like user id, db connections.
    2. **Store**: BaseStore instance used for long-term memory
    3. **Stream writer**: an object used for streaming information via the "custom" stream mode
    4. **Execution info**: identity and retry information for the current execution (thread ID, run ID, attempt number)

    <img src="../../assets/agent_runtime.png" width="600" height="300">


In [9]:
import os
import json
from dotenv import load_dotenv
from typing import Literal
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore


load_dotenv()

True

In [3]:
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable is not set.")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

model = ChatOpenAI(model="gpt-5-nano")

### Runtime Inside Tools : ToolRuntime
<img src="../../assets/runtime_flow.png" width="600" height="300">

In [11]:
class CustomerContext(BaseModel):
  user_id:str = Field(..., description="The unique identifier for the user.")

loyalty_store=InMemoryStore()

In [12]:
@tool
def fetch_customer_preferences(runtime: ToolRuntime[CustomerContext])-> str:
  "Fetch the customer's Saved Preferences from a long term Memory"
  user_id=runtime.context.user_id
  preferences = "No Preferences"

  if runtime.store:
    if memory := runtime.store.get(('users'),user_id):
      preferences = memory.value['preferences']

  return preferences

In [ ]:
pref_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[fetch_customer_preferences],
    context_schema=CustomerContext,
    store=loyalty_store,
)

In [13]:
from langchain.agents.middleware import before_model, after_model

## Middleware
* NodeStyle
* wrap_style_hoooks (wrapp_model_call)

#### Node Style Hooks

In [ ]:
@before_model
def log_before_model():   
    print(f"Input to model")